In [20]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

print("🚀 Starting Data Processing Workflow...")

# =========================================================================
# 1. LOAD THE REAL FILE PIPELINES
# =========================================================================
try:
    df_customers = pd.read_csv('customers.csv', encoding='latin1')
    df_orders = pd.read_csv('orders.csv', encoding='latin1')
    df_items = pd.read_csv('order_items.csv', encoding='latin1')
    df_payments = pd.read_csv('payments.csv', encoding='latin1')
    df_products = pd.read_csv('products.csv', encoding='latin1')
    df_sellers = pd.read_csv('sellers.csv', encoding='latin1')
    print("✅ All 6 dataset files loaded successfully!")
except FileNotFoundError as e:
    print(f"❌ Error: Missing file in directory. Detail: {e}")
    print("Ensure all CSV filenames match Olist specifications and sit in this workspace.")
    raise

# Clean dates and filter out invalid/unfulfilled order entries upfront
df_orders['order_purchase_timestamp'] = pd.to_datetime(df_orders['order_purchase_timestamp'])
df_clean_orders = df_orders[~df_orders['order_status'].isin(['canceled', 'unavailable'])].copy()


#BASIC PROBLEMS:
# 1. List all unique cities where customers are located.
unique_cities = df_customers['customer_city'].unique()
print(unique_cities)
print("Total Rows:",len(unique_cities))

# 2. Count the number of orders placed in 2017

orders_2017 = df_orders[df_orders['order_purchase_timestamp'].dt.year == 2017]['order_id'].count()
print("Total Orders in 2017:", orders_2017)

#3. Find the total sales per category.

merged = pd.merge(df_items, df_products, on='product_id')
sales_per_cat = merged.groupby('product_category')['price'].sum().round(2).reset_index()
sales_per_cat = sales_per_cat.sort_values(by='price', ascending=False)
print(sales_per_cat)

#4. Calculate the percentage of orders that were paid in installments.

installment_orders = df_payments[df_payments['payment_installments'] > 1]['order_id'].nunique()
total_orders = df_payments['order_id'].nunique()
installment_pct = (installment_orders / total_orders) * 100

print(f"Installment Orders: {installment_orders}")
print(f"Total Orders: {total_orders}")
print(f"Installment Percentage: {installment_pct:.2f}%")

#5. Count the number of customers from each state.


state_counts = df_customers['customer_state'].value_counts().reset_index()
state_counts.columns = ['customer_state', 'customer_count']
print(state_counts)

#INTERMEDIATE PROBLEMS
#1. Calculate the number of orders per month in 2018

orders_2018 = df_orders[df_orders['order_purchase_timestamp'].dt.year == 2018].copy()
orders_2018['month'] = orders_2018['order_purchase_timestamp'].dt.month
monthly_orders = orders_2018.groupby('month')['order_id'].count().reset_index()
print(monthly_orders)

#2. Find the average number of products per order, grouped by customer city.

item_counts = df_items.groupby('order_id')['order_item_id'].count().reset_index()
merged_city = df_orders.merge(df_customers, on='customer_id').merge(item_counts, on='order_id')

avg_prod_city = merged_city.groupby('customer_city').agg(
    average_prod_per_order=('order_item_id', 'mean'),
    total_orders_placed=('order_id', 'count')
).reset_index()

avg_prod_city['average_prod_per_order'] = avg_prod_city['average_prod_per_order'].round(2)
avg_prod_city = avg_prod_city.sort_values(by='average_prod_per_order', ascending=False)

print(avg_prod_city.to_string(index=False))

#3. Calculate the percentage of total revenue contributed by each product category
merged = df_items.merge(df_products, on='product_id')
total_rev = df_items['price'].sum()
cat_rev = (merged.groupby('product_category')['price'].sum() / total_rev * 100).round(2).reset_index()
print(cat_rev.sort_values(by='price', ascending=False))

 #4. Identify the correlation between product price and the number of times a product has been purchased.
prod_stats = df_items.groupby('product_id').agg(
    purchase_count=('order_item_id', 'count'),
    avg_price=('price', 'mean')
)
corr = prod_stats['purchase_count'].corr(prod_stats['avg_price'])
print(f"Correlation: {corr:.4f}")

#5. Calculate the total revenue generated by each seller, and rank them by revenue.
seller_rev = df_items.groupby('seller_id')['price'].sum().round(2).reset_index()
seller_rev['revenue_rank'] = seller_rev['price'].rank(ascending=False, method='min').astype(int)
print(seller_rev.sort_values('revenue_rank'))

#ADVANCED PROBLEMS
#1. Calculate the moving average of order values for each customer over their order history.
cust_orders = df_customers.merge(df_orders, on='customer_id').merge(df_payments, on='order_id')
order_vals = cust_orders.groupby(['customer_unique_id', 'order_id', 'order_purchase_timestamp'])['payment_value'].sum().reset_index()
order_vals = order_vals.sort_values(['customer_unique_id', 'order_purchase_timestamp'])
order_vals['moving_avg'] = order_vals.groupby('customer_unique_id')['payment_value'].expanding().mean().round(2).reset_index(level=0, drop=True)
print(order_vals.head())


#2. Calculate the cumulative sales per month for each year.
sales_df = df_orders.merge(df_payments, on='order_id')
sales_df['year'] = sales_df['order_purchase_timestamp'].dt.year
sales_df['month'] = sales_df['order_purchase_timestamp'].dt.month
monthly = sales_df.groupby(['year', 'month'])['payment_value'].sum().reset_index()
monthly['cumulative_sales'] = monthly.groupby('year')['payment_value'].cumsum().round(2)
print(monthly.head(15))

#3. Calculate the year-over-year growth rate of total sales.
yearly = sales_df.groupby('year')['payment_value'].sum().reset_index()
yearly['yoy_growth_pct'] = (yearly['payment_value'].pct_change() * 100).round(2)
print(yearly)

#4. Calculate the retention rate of customers, defined as the percentage of customers who make another purchase
#within 6 months of their first purchase

df_orders['order_purchase_timestamp'] = pd.to_datetime(df_orders['order_purchase_timestamp'])
cust_ord = df_customers.merge(df_orders, on='customer_id')

first_purch = cust_ord.groupby('customer_id')['order_purchase_timestamp'].min().reset_index()
first_purch.columns = ['customer_id', 'first_purchase']

merged_ret = cust_ord.merge(first_purch, on='customer_id')
retained = merged_ret[
    (merged_ret['order_purchase_timestamp'] > merged_ret['first_purchase']) &
    (merged_ret['order_purchase_timestamp'] <= merged_ret['first_purchase'] + pd.DateOffset(months=6))
]['customer_id'].nunique()

total_cust = first_purch['customer_id'].nunique()
retention_pct = (retained / total_cust) * 100

print(f"Retained Customers: {retained}")
print(f"Total Customers: {total_cust}")
print(f"Retention Rate: {retention_pct:.2f}%")


#5. Identify the top 3 customers who spent the most money in each year.

full_df = df_customers.merge(df_orders, on='customer_id').merge(df_items, on='order_id')
full_df['year'] = pd.to_datetime(full_df['order_purchase_timestamp']).dt.year

yearly_cust = full_df.groupby(['year', 'customer_unique_id']).agg(
    total_spent=('price', 'sum'),
    total_orders=('order_id', 'nunique')
).reset_index()

# Convert to integer cents before ranking so floating-point precision never breaks ties
yearly_cust['spent_cents'] = (yearly_cust['total_spent'] * 100).round().astype(int)
yearly_cust['total_spent'] = yearly_cust['spent_cents'] / 100.0

yearly_cust['spending_rank'] = yearly_cust.groupby('year')['spent_cents'].rank(method='dense', ascending=False).astype(int)

top_3 = yearly_cust[yearly_cust['spending_rank'] <= 3][['year', 'spending_rank', 'customer_unique_id', 'total_spent', 'total_orders']]
top_3 = top_3.sort_values(['year', 'spending_rank', 'customer_unique_id'])

print(top_3.to_string(index=False, float_format=lambda x: f"{x:.2f}"))


# =========================================================================
# 2. DATA PROCESSING FOR METRIC 1: CUMULATIVE SALES PER MONTH
# =========================================================================
df_clean_orders['year'] = df_clean_orders['order_purchase_timestamp'].dt.year
df_clean_orders['month_num'] = df_clean_orders['order_purchase_timestamp'].dt.month
df_clean_orders['month_name'] = df_clean_orders['order_purchase_timestamp'].dt.strftime('%b')

sales_merged = df_clean_orders.merge(df_items, on='order_id', how='inner')
monthly_sales = sales_merged.groupby(['year', 'month_num', 'month_name'])['price'].sum().reset_index()
monthly_sales['cumulative_sales'] = monthly_sales.groupby('year')['price'].cumsum() / 1e6 # Convert to millions

# Pivot table for Line Chart plotting
df_cum_pivot = monthly_sales.pivot(index='month_num', columns='year', values='cumulative_sales')
month_labels = {1:'Jan', 2:'Feb', 3:'Mar', 4:'Apr', 5:'May', 6:'Jun', 7:'Jul', 8:'Aug', 9:'Sep', 10:'Oct', 11:'Nov', 12:'Dec'}
df_cum_pivot.index = df_cum_pivot.index.map(month_labels)

# =========================================================================
# 3. DATA PROCESSING FOR METRIC 2: MARKET SHARE CONTRIBUTION BY CATEGORY
# =========================================================================
cat_merged = df_items.merge(df_products, on='product_id', how='inner')
category_revenue = cat_merged.groupby('product_category')['price'].sum().reset_index()
total_marketplace_revenue = category_revenue['price'].sum()
category_revenue['contribution_pct'] = (category_revenue['price'] / total_marketplace_revenue) * 100

# Format labels and pull top 10 categories
category_revenue['product_category'] = category_revenue['product_category'].str.replace('_', ' ').str.title()
top_10_categories = category_revenue.sort_values(by='contribution_pct', ascending=True).tail(10)

# =========================================================================
# 4. DATA PROCESSING FOR METRIC 3: PRICE VS. PURCHASES CORRELATION
# =========================================================================
product_metrics = df_items.groupby('product_id').agg(
    avg_price=('price', 'mean'),
    purchase_count=('order_id', 'count')
).reset_index()

# Filter out high extreme pricing anomalies over 1000 BRL to protect chart bounds
product_metrics_clean = product_metrics[product_metrics['avg_price'] <= 1000]
pearson_r = product_metrics_clean['avg_price'].corr(product_metrics_clean['purchase_count'], method='pearson')
spearman_rho = product_metrics_clean['avg_price'].corr(product_metrics_clean['purchase_count'], method='spearman')

# =========================================================================
# 5. DATA PROCESSING FOR METRIC 4: 6-MONTH RETENTION WINDOW DIAGNOSTIC
# =========================================================================
user_orders = df_clean_orders.merge(df_customers, on='customer_id', how='inner')
user_orders['first_purchase'] = user_orders.groupby('customer_unique_id')['order_purchase_timestamp'].transform('min')
user_orders['days_since_first'] = (user_orders['order_purchase_timestamp'] - user_orders['first_purchase']).dt.days

user_orders['is_retained_6m'] = (user_orders['days_since_first'] > 0) & (user_orders['days_since_first'] <= 180)
retention_summary = user_orders.groupby('customer_unique_id')['is_retained_6m'].max().reset_index()

retained_count = retention_summary['is_retained_6m'].sum()
churned_count = len(retention_summary) - retained_count
retention_percentages = [churned_count, retained_count]

# =========================================================================
# 6. INITIALIZE MULTI-PLOT CANVAS DASHBOARD
# =========================================================================
plt.style.use('default')
plt.rcParams.update({'font.size': 10, 'axes.grid': True, 'grid.linestyle': '--', 'grid.alpha': 0.5})

fig, axes = plt.subplots(2, 2, figsize=(20, 14))
fig.suptitle("Online Shopping Analytical Dashboard", fontsize=22, fontweight='bold', y=0.98)

# PLOT 1: Cumulative Sales Trend Per Year
df_cum_pivot.plot(kind='line', ax=axes[0, 0], marker='o', linewidth=2.5, color=['#4682B4', '#FF8C00', '#2E8B57'])
axes[0, 0].set_title("Cumulative Revenue Profile (Year-over-Year Running Total)", fontweight='bold', pad=10)
axes[0, 0].set_ylabel("Cumulative Value (Millions BRL)")
axes[0, 0].set_xlabel("Calendar Month")
axes[0, 0].legend(title="Year Context")

# PLOT 2: Category Market Share Breakdown
top_10_categories.plot(kind='barh', x='product_category', y='contribution_pct', ax=axes[0, 1], color='#34495E', legend=False)
for index, value in enumerate(top_10_categories['contribution_pct']):
    axes[0, 1].text(value + 0.1, index, f" {value:.1f}%", va='center', fontweight='bold', fontsize=9)
axes[0, 1].set_title("Market Share Concentration: Revenue Contribution by Category", fontweight='bold', pad=10)
axes[0, 1].set_xlabel("Percentage of Total Marketplace Sales (%)")
axes[0, 1].set_ylabel("")
axes[0, 1].set_xlim(0, top_10_categories['contribution_pct'].max() + 2)

# PLOT 3: Product Price vs. Transaction Sales Volumes
axes[1, 0].scatter(product_metrics_clean['avg_price'], product_metrics_clean['purchase_count'], alpha=0.3, color='#16A085', edgecolors='w', s=25)
m, b = np.polyfit(product_metrics_clean['avg_price'], product_metrics_clean['purchase_count'], 1)
axes[1, 0].plot(product_metrics_clean['avg_price'], m*product_metrics_clean['avg_price'] + b, color='#C0392B', linewidth=2.5)
axes[1, 0].set_title("Elasticity Assessment: Price vs. Total Purchases Count", fontweight='bold', pad=10)
axes[1, 0].set_xlabel("Average Product Price (BRL)")
axes[1, 0].set_ylabel("Lifetime Purchases Volume")
axes[1, 0].set_xlim(0, 1000)
axes[1, 0].set_ylim(0, product_metrics_clean['purchase_count'].max() + 5)
stats_box = f"Pearson r: {pearson_r:.3f}\nSpearman ρ: {spearman_rho:.3f}"
axes[1, 0].text(0.65, 0.82, stats_box, transform=axes[1, 0].transAxes, fontweight='bold', fontsize=11, bbox=dict(facecolor='white', alpha=0.9, edgecolor='gray'))

# PLOT 4: 6-Month Retention Window Diagnostic
axes[1, 1].pie(retention_percentages, labels=['One-and-Done\n(Churned)', 'Repeat Customer\n(Retained <= 6 Mo)'],
               autopct='%1.2f%%', startangle=140, colors=['#E67E22', '#2980B9'], explode=(0, 0.2), textprops={'fontweight':'bold', 'fontsize':11})
axes[1, 1].set_title("Marketplace Stickiness: 6-Month Customer Retention Rate", fontweight='bold', pad=10)
axes[1, 1].grid(False) # Turn off grid specifically for the pie layout canvas

# Clean up window allocations and paint the screen
plt.tight_layout()
print("📊 Rendering final analytical dashboard visualizations...")
plt.show()





Retained Customers: 0
Total Customers: 99441
Retention Rate: 0.00%
